# Pelajaran 11 - Protokol Agen-ke-Agen (A2A)


## Pengaturan


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## What is the A2A Protocol?

The **Agent-to-Agent (A2A) protocol** is an open standard that enables AI agents to communicate,
discover each other, and collaborate — even when they are built on different frameworks or hosted
by different services.

Key concepts:

- **Discovery** – Agents publish an *Agent Card* that describes their capabilities, making it
  easy for other agents (or orchestrators) to find the right specialist for a task.
- **Message Passing** – Agents exchange structured messages through a common protocol, so a
  request from one agent can be understood and fulfilled by another regardless of its internal
  implementation.
- **Task Lifecycle** – A2A defines states such as *submitted*, *working*, *completed*, and
  *failed*, giving the orchestrator full visibility into how a delegated task is progressing.

In this lesson we simulate A2A-style collaboration by wiring three specialized travel agents
into a workflow where each agent contributes its expertise and passes results to the next.


## Membuat Agen Perjalanan Khusus


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Kolaborasi Multi-Agen melalui Alur Kerja

Kami menghubungkan tiga agen ke dalam alur kerja berurutan yang mencerminkan pengiriman pesan A2A:

1. **CurrencyExchangeAgent** menerima permintaan pengguna dan menghasilkan panduan mata uang.
2. **ActivityPlannerAgent** menerima konteks yang diperkaya dan menambahkan rekomendasi aktivitas.
3. **TravelManagerAgent** mensintesis kedua masukan menjadi ringkasan perjalanan akhir.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Memahami A2A dalam Produksi

Dalam lingkungan produksi, protokol A2A membuka skenario lintas-layanan yang kuat:

| Kemampuan | Deskripsi |
|---|---|
| **Interop lintas kerangka kerja** | Agen yang dibuat dengan satu kerangka kerja dapat mendelegasikan tugas ke agen yang dibuat dengan kerangka kerja lain yang mematuhi A2A, memungkinkan interoperabilitas lintas organisasi yang sejati. |
| **Batas layanan** | Agen bisa berada di layanan mikro terpisah, wilayah cloud, atau bahkan organisasi yang berbeda sekalipun tetap dapat berkolaborasi dengan mulus. |
| **Penemuan dinamis** | Sebuah pengatur dapat melakukan query pada registri Kartu Agen saat runtime untuk menemukan spesialis yang paling sesuai untuk sub-tugas tertentu. |
| **Streaming & notifikasi push** | A2A mendukung Server-Sent Events (SSE) untuk pembaruan progres waktu nyata dan notifikasi push untuk tugas yang berjalan lama. |

Alur kerja yang kami buat di atas adalah versi sederhana dalam-proses dari pola ini. Dalam
penyebaran nyata, setiap agen akan mengekspos endpoint HTTP, menerbitkan Kartu Agen,
dan berkomunikasi melalui protokol A2A JSON-RPC.


## Summary

In this lesson you learned:

1. **What the A2A protocol is** — an open standard for agent-to-agent discovery, messaging,
   and task management.
2. **How to create specialized agents** — a Currency Exchange agent, an Activity Planner agent,
   and a Travel Manager orchestrator.
3. **How to wire agents into a workflow** — using `WorkflowBuilder` to model sequential
   message passing between agents.
4. **How A2A works in production** — enabling cross-framework, cross-service collaboration
   with dynamic discovery and streaming updates.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berupaya untuk mencapai akurasi, harap diketahui bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang sah. Untuk informasi penting, disarankan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau penafsiran yang keliru yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
